# Phase 1: Dataset Ingestion, Verification & Exploratory Data Analysis (EDA)
## AI-Powered Fake Job and Recruitment Scam Detection System

**Project Context**: Final-Year B.Tech CSE (AI & ML) Academic Project  
**Objective**: Ingest, verify cryptographic integrity, inspect schema, analyze missingness, examine severe class imbalance, profile text fields, explore fraud indicators, and evaluate potential train-test data leakage.

> **Academic Reproducibility Notice**: All figures and statistics in this notebook are computed dynamically from the raw immutable dataset using random seed `42`.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from config.settings import get_config
from src.data.loader import load_raw_dataset, get_file_metadata
from src.data.inspection import (
    inspect_schema,
    compute_basic_statistics,
    analyze_target,
    analyze_duplicates,
    analyze_missing_values,
)
from src.data.text_stats import analyze_all_text_columns
from src.data.leakage_checker import review_data_leakage
from src.data.eda_plots import generate_all_eda_figures

print("Environment and libraries loaded successfully.")

## 1. Raw Dataset Ingestion & Cryptographic Integrity Check

In [ ]:
config = get_config()
raw_path = config.raw_dataset_path

# Load raw dataset without modifying the file
df, metadata = load_raw_dataset(raw_path)

print(f"Dataset Filename : {metadata['filename']}")
print(f"File Size        : {metadata['size_mb']} MB ({metadata['size_bytes']:,} bytes)")
print(f"SHA-256 Hash     : {metadata['sha256']}")
print(f"Shape            : {df.shape[0]:,} rows x {df.shape[1]} columns")

## 2. Schema Verification vs Canonical EMSCAD Reference

In [ ]:
schema_info = inspect_schema(df)
print(f"Exact EMSCAD Schema Match: {schema_info['is_emscad_schema_match']}")
print(f"Matching Columns ({schema_info['matching_columns_count']}/{schema_info['actual_column_count']}):")
for i, col in enumerate(schema_info['actual_columns'], 1):
    print(f"  {i:02d}. {col}")

## 3. Target Variable Analysis (`fraudulent`)

In [ ]:
target_stats = analyze_target(df, target_col=config.target_column)

print("=" * 50)
print("TARGET CLASS DISTRIBUTION")
print("=" * 50)
print(f"Legitimate Postings (0) : {target_stats['legitimate_count (0)']:,} ({target_stats['legitimate_percentage']:.2f}%)")
print(f"Fraudulent Postings (1) : {target_stats['fraudulent_count (1)']:,} ({target_stats['fraudulent_percentage']:.2f}%)")
print(f"Imbalance Ratio         : {target_stats['imbalance_ratio_legit_to_fraud']}")
print("=" * 50)

## 4. Missing Values & Duplicate Inspection

In [ ]:
missing_info = analyze_missing_values(df)
dup_info = analyze_duplicates(df)

print(f"Exact Duplicate Rows : {dup_info['exact_duplicate_rows']}")
print(f"Duplicate Job IDs    : {dup_info['duplicate_job_ids']}")
print(f"Duplicate Titles     : {dup_info['duplicate_titles']:,}")
print(f"Duplicate Desc.      : {dup_info['duplicate_descriptions']:,}")

print("\nTop 5 Most Incomplete Columns:")
for col in missing_info['columns_summary'][:5]:
    print(f"  - {col['column']:<22} : {col['missing_count']:,} missing ({col['missing_percentage']:.1f}%)")

## 5. Unstructured Text Fields Character & Word Count Profiling

In [ ]:
text_info = analyze_all_text_columns(df, target_col=config.target_column)

summary_rows = []
for field in ["title", "company_profile", "description", "requirements", "benefits", "composite_job_text"]:
    if field in text_info:
        ov = text_info[field]["overall"]
        st = text_info[field]["class_stratified"]
        summary_rows.append({
            "Field": field,
            "Populated (%)": f"{ov['populated_pct']:.1f}%",
            "Legit Mean Chars": st.get("legitimate", {}).get("char_length", {}).get("mean", 0),
            "Fraud Mean Chars": st.get("fraudulent", {}).get("char_length", {}).get("mean", 0),
            "Legit Mean Words": st.get("legitimate", {}).get("word_count", {}).get("mean", 0),
            "Fraud Mean Words": st.get("fraudulent", {}).get("word_count", {}).get("mean", 0),
        })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

## 6. Generate and Save Publication-Quality Figures

In [ ]:
figures_dir = config.figures_dir
saved_plots = generate_all_eda_figures(df, figures_dir, target_col=config.target_column)

print(f"Generated {len(saved_plots)} publication figures under '{figures_dir}':")
for p in saved_plots:
    print(f"  [OK] {Path(p).name}")

## 7. Data Leakage Review & Preprocessing Recommendations

In [ ]:
leakage = review_data_leakage(df, target_col=config.target_column)
leakage_df = pd.DataFrame(leakage)
print(leakage_df[["column", "potential_leakage", "risk_level", "recommended_action"]].to_string(index=False))

print("\nPhase 1 Dataset Ingestion, Verification & EDA complete. Dataset is READY FOR PHASE 2.")